# QLoRA Fine-Tuning: TelkomNusa-SLM-3B & TelkomNusa-LLM-7B

Trains the 4-bit NormalFloat (NF4) QLoRA adapters for `Qwen/Qwen2.5-3B-Instruct` (`TelkomNusa-SLM-3B`) and `Qwen/Qwen2.5-7B-Instruct` (`TelkomNusa-LLM-7B`) on the balanced `Rislantrs/TelkomNusa-SFT-ID` corpus (3,129 training / 381 validation dialogues across 23 BSS tools).

Hyperparameters match Table 4 in the manuscript:
- Quantization: 4-bit NF4 with double quantization
- LoRA configuration: rank $r = 16$, scaling $\alpha = 32$, dropout $= 0$, all 7 linear layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`)
- Optimization: `adamw_8bit`, learning rate $2 \times 10^{-4}$ with cosine schedule, 20 warmup steps, 3 epochs, effective batch size 16, `max_seq_length = 2048`


## 1. Environment Setup and GPU Verification


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets matplotlib huggingface_hub


In [ ]:
import gc
import json
import os
import time
import zipfile

import matplotlib.pyplot as plt
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, login, whoami
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

torch.cuda.empty_cache()
gc.collect()

assert torch.cuda.is_available(), "CUDA GPU is required for Unsloth 4-bit QLoRA training."
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU: {gpu_name} ({total_vram_gb:.2f} GB VRAM)")


## 2. Hugging Face Hub Authentication and Target Configuration


In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
HF_USERNAME = "Rislantrs"
HF_AUTHENTICATED = False

if HF_TOKEN and not HF_TOKEN.startswith("hf_your"):
    try:
        login(token=HF_TOKEN)
        user_info = whoami(token=HF_TOKEN)
        HF_USERNAME = user_info.get("name", user_info.get("username", HF_USERNAME))
        HF_AUTHENTICATED = True
        print(f"Authenticated on Hugging Face Hub as: {HF_USERNAME}")
    except Exception as exc:
        print(f"Hugging Face login skipped ({exc}); adapters will be saved locally only.")
else:
    print("HF_TOKEN not set; running in local save mode.")

# Switch between "3B" (TelkomNusa-SLM-3B) and "7B" (TelkomNusa-LLM-7B).
MODEL_CHOICE = "3B"
MAX_SEQ_LENGTH = 2048

if MODEL_CHOICE.upper() == "3B":
    BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
    HF_REPO_NAME = f"{HF_USERNAME}/telecom-troubleshooter-qwen2.5-3b-balanced-lora"
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4
else:
    BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
    HF_REPO_NAME = f"{HF_USERNAME}/telecom-troubleshooter-qwen2.5-7b-balanced-lora"
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 8

print(f"Target base model : {BASE_MODEL_NAME}")
print(f"Output repo ID    : {HF_REPO_NAME}")
print(f"Effective batch   : {BATCH_SIZE * GRAD_ACCUM_STEPS} | Max seq length: {MAX_SEQ_LENGTH}")


## 3. Load Balanced SFT Corpus (`TelkomNusa-SFT-ID`)

Searches local paths or mounted ZIP archives first, and otherwise loads `Rislantrs/TelkomNusa-SFT-ID` directly from Hugging Face Hub.


In [ ]:
SEARCH_PATHS = [
    "/content",
    "/kaggle/input",
    "dataset_sft_telko_indonesia",
    "..",
    ".",
]

for sp in list(SEARCH_PATHS):
    if os.path.exists(sp):
        for fname in os.listdir(sp):
            if fname.endswith(".zip") and "sft" in fname.lower():
                with zipfile.ZipFile(os.path.join(sp, fname), "r") as zf:
                    zf.extractall("./dataset_extracted")
                SEARCH_PATHS.insert(0, "./dataset_extracted")
                break

TRAIN_FILE, VAL_FILE = None, None
for base_dir in SEARCH_PATHS:
    if not os.path.exists(base_dir):
        continue
    for root, _, files in os.walk(base_dir):
        for fname in files:
            low = fname.lower()
            if not low.endswith(".jsonl"):
                continue
            if "train" in low and TRAIN_FILE is None:
                TRAIN_FILE = os.path.join(root, fname)
            elif ("val" in low or "eval" in low) and VAL_FILE is None:
                VAL_FILE = os.path.join(root, fname)

if TRAIN_FILE and VAL_FILE:
    print(f"Loading local SFT JSONL files: {TRAIN_FILE}, {VAL_FILE}")
    with open(TRAIN_FILE, "r", encoding="utf-8") as f:
        train_data = [json.loads(line) for line in f if line.strip()]
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        val_data = [json.loads(line) for line in f if line.strip()]
else:
    print("Local JSONL files not found; loading Rislantrs/TelkomNusa-SFT-ID from Hugging Face Hub...")
    hf_sft = load_dataset("Rislantrs/TelkomNusa-SFT-ID")
    train_data = list(hf_sft["train"])
    val_data = list(hf_sft["validation"])

mt_count = sum(1 for d in train_data if sum(1 for m in d.get("messages", []) if m.get("role") == "user") > 1)
print(f"Train samples: {len(train_data)} (Single-Turn: {len(train_data) - mt_count}, Multi-Turn: {mt_count})")
print(f"Val samples  : {len(val_data)}")


## 4. Initialize 4-Bit NF4 Base Model and Attach All-Linear LoRA Adapter


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"Allocated GPU memory: {torch.cuda.memory_allocated(0) / (1024 ** 3):.2f} GB")


## 5. Format Dialogues to Qwen2.5 ChatML and Run QLoRA Fine-Tuning


In [ ]:
def format_sft_sample(sample: dict) -> str:
    messages = sample["messages"]
    tools = sample.get("tools") or None
    if tools:
        return tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=False)
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


train_dataset = Dataset.from_list([{"text": format_sft_sample(s)} for s in train_data])
val_dataset = Dataset.from_list([{"text": format_sft_sample(s)} for s in val_data])

BASE_OUTPUT_DIR = "/content" if os.path.exists("/content") else ("/kaggle/working" if os.path.exists("/kaggle/working") else ".")
OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, f"qlora_telkomnusa_{MODEL_CHOICE.lower()}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    num_train_epochs=3,
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

start_time = time.time()
train_result = trainer.train()
print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes.")


## 6. Loss Curve Diagnostics, Adapter Export, and Inference Sanity Check


In [ ]:
train_steps, train_losses, eval_steps, eval_losses = [], [], [], []
for entry in trainer.state.log_history:
    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    if "eval_loss" in entry and "step" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])

plt.figure(figsize=(9, 4.5), dpi=300)
if train_steps:
    plt.plot(train_steps, train_losses, label=f"Training Loss ({MODEL_CHOICE.upper()})", color="#1f77b4", lw=2)
if eval_steps:
    plt.plot(eval_steps, eval_losses, label=f"Validation Loss ({MODEL_CHOICE.upper()})", color="#d62728", lw=2, linestyle="--")
plt.title(f"QLoRA Convergence ({BASE_MODEL_NAME})")
plt.xlabel("Training Steps")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT_DIR, f"loss_curve_{MODEL_CHOICE.lower()}.png"), dpi=300)
plt.show()

LOCAL_ADAPTER_DIR = os.path.join(BASE_OUTPUT_DIR, f"telecom-troubleshooter-qwen2.5-{MODEL_CHOICE.lower()}-balanced-lora")
os.makedirs(LOCAL_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"Saved adapter and tokenizer to: {LOCAL_ADAPTER_DIR}")

if HF_AUTHENTICATED and HF_TOKEN:
    model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    print(f"Pushed adapter to: https://huggingface.co/{HF_REPO_NAME}")


In [ ]:
FastLanguageModel.for_inference(model)

prompt_st = [
    {"role": "system", "content": "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."},
    {"role": "user", "content": "Tolong cekkan rincian tunggakan tagihan nomor saya 081299887766."},
]
inp_st = tokenizer.apply_chat_template(prompt_st, tokenize=False, add_generation_prompt=True)
in_ids_st = tokenizer(inp_st, return_tensors="pt").to("cuda")
with torch.no_grad():
    out_st = model.generate(**in_ids_st, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print("Single-Turn Sanity Check:")
print(tokenizer.decode(out_st[0][in_ids_st.input_ids.shape[1] :], skip_special_tokens=False))

prompt_mt = [
    {"role": "system", "content": "Anda adalah asisten AI resmi Customer Care TelkomNusa. Panggil tool yang sesuai jika diperlukan."},
    {"role": "user", "content": "Bisa cek tagihan saya? Nomor saya 081299887766."},
    {"role": "assistant", "content": '<tool_call>\n{"name": "GET__rate_unpaid-bills", "arguments": {"svcMgmtNum": "081299887766"}}\n</tool_call>'},
    {"role": "tool", "name": "GET__rate_unpaid-bills", "content": '{"resultCode": "0000", "billAmt": 150000, "status": "Belum Lunas"}'},
    {"role": "assistant", "content": "Tagihan Anda adalah sebesar Rp150.000. Ada yang bisa dibantu lagi?"},
    {"role": "user", "content": "Sekalian cek sisa kuota data internet saya dong."},
]
inp_mt = tokenizer.apply_chat_template(prompt_mt, tokenize=False, add_generation_prompt=True)
in_ids_mt = tokenizer(inp_mt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out_mt = model.generate(**in_ids_mt, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print("\nMulti-Turn Sanity Check:")
print(tokenizer.decode(out_mt[0][in_ids_mt.input_ids.shape[1] :], skip_special_tokens=False))
